# BRFSS Diabetes Model Evaluation

This notebook provides a Google Colab workflow for the project. It analyzes self-reported prevalent diabetes and compares models; it does not diagnose diabetes or predict future onset. The analysis requires the CDC SAS XPT files for 2024 and 2025. Do not upload personal health information or files containing direct identifiers.

**Workflow:** prepare the repository, install dependencies, place the CDC files in the data folder, run tests and analysis, then review the results.

## 1. Set up the project

After the updates are merged on GitHub, run the next cell to clone the repository. If you already have a local copy under `/content`, set `PROJECT_DIR` to that location instead.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/alrashedinaifrr/brfss-diabetes-model-evaluation.git'
PROJECT_DIR = Path('/content/brfss-diabetes-model-evaluation')
if not (PROJECT_DIR / 'run_v11.py').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
assert (PROJECT_DIR / 'run_v11.py').exists(), 'Could not find run_v11.py in the project root.'
print('Project:', PROJECT_DIR)

In [ ]:
%cd /content/brfss-diabetes-model-evaluation
%pip install -q -r requirements.txt

## 2. Prepare the BRFSS files

Download the SAS Transport files from the official CDC page and place them in a Google Drive folder named `brfss_data`. Update the path below if needed. The files must be named `LLCP2024.XPT` and `LLCP2025.XPT`. The pipeline reads the files in chunks to limit peak parsing memory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/brfss_data')
required = [DATA_DIR / 'LLCP2024.XPT', DATA_DIR / 'LLCP2025.XPT']
for path in required:
    print(('✓' if path.is_file() else '✗'), path)
if not all(p.is_file() for p in required):
    raise FileNotFoundError('Place both XPT files in the brfss_data folder on Drive or update DATA_DIR.')

## 3. Run code checks and the analysis

Outputs include weighted and unweighted evaluation, exploratory decision-threshold scenarios, BMI subgroup analysis, calibration, and sensitivity comparisons. Survey weights alone do not establish design-correct confidence intervals.

In [ ]:
subprocess.run(['python', '-m', 'pytest', '-q', 'test_v11.py'], cwd=PROJECT_DIR, check=True)
OUTPUT_DIR = Path('/content/brfss_results')
subprocess.run([
    'python', str(PROJECT_DIR / 'run_v11.py'),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
], check=True)

In [ ]:
import pandas as pd
from IPython.display import display

summary_path = OUTPUT_DIR / 'model_comparison_2025.csv'
display(pd.read_csv(summary_path).head(20))
print('Output files:')
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file(): print(path.relative_to(OUTPUT_DIR))

In [ ]:
from IPython.display import display, Image
for path in sorted((OUTPUT_DIR / 'figures').glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/brfss_results', 'zip', OUTPUT_DIR)
print('Results archive:', archive)
files.download(archive)

## Interpreting the results

- Metrics recorded from an earlier version do not replace rerunning this version; results may change with the code or input data.
- The 2025 evaluation data were used during development, so they are not an untouched external test set.
- This is exploratory research, not a medical tool or screening recommendation. See `TECHNICAL_REPORT.md` and `VALIDATION_STATUS.md` for methods and limitations.